In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Uncomment to install Libraries (if not installed)
#!pip3 install openpyxl

In [10]:
# Read the Excel files and save in one DataFrame
allowed_ext = ".xlsx"
excel_dir_path = "analysis/"
files = os.listdir(excel_dir_path)
files = sorted(files)

rename_map = {
    "Nodes-Number of edge.": "Nodes-Edges",
    "Nodes-Number of edge. (Fitting)":"Nodes-Edges(Fit)",
    "Nodes-Average degree":"Nodes-Degree",
    "Nodes-Average degree (Fitting)":"Nodes-Degree(Fit)",
    "Nodes-Network diamet.":"Nodes-Diameter",
    "Nodes-Network diamet. (Fitting)":"Nodes-Diameter(Fit)",
    "Nodes-Graph density":"Nodes-Density",
    "Nodes-Graph density (Fitting)":"Nodes-Density(Fit)",
    "Nodes-Average betwee.":"Nodes-BC",
    "Nodes-Average betwee. (Fitting)":"Nodes-BC(Fit)",
    "Nodes-Average eigenv.":"Nodes-EC",
    "Nodes-Average eigenv. (Fitting)":"Nodes-EC(Fit)",
    "Nodes-Average closen.":"Nodes-CC",
    "Nodes-Average closen. (Fitting)":"Nodes-CC(Fit)",
    "Nodes-Assortativity .":"Nodes-ASC",
    "Nodes-Assortativity . (Fitting)":"Nodes-ASC(Fit)",
    "Nodes-Average cluste.":"Nodes-ACC",
    "Nodes-Average cluste. (Fitting)":"Nodes-ACC(Fit)",
    "Nodes-Global efficie.":"Nodes-GE",
    "Nodes-Global efficie. (Fitting)":"Nodes-GE(Fit)",
    "Nodes-Wiener Index":"Nodes-WI",
    "Nodes-Wiener Index (Fitting)":"Nodes-WI(Fit)",
}

all_sheets = {}
for a_file in files:
    if a_file.endswith(allowed_ext):
        # Get the Excel file and load its contents
        file_path = os.path.join(excel_dir_path, a_file)
        file_sheets = pd.read_excel(file_path, sheet_name=None)

        # Append Excel data to one place
        for sheet_name, df in file_sheets.items():
            # Rename if sheet_name exists in mapping
            new_name = rename_map.get(sheet_name, sheet_name)

            # Add Material column with file name (without extension)
            df = df.copy()
            df["Material"] = os.path.splitext(a_file)[0]

            if new_name not in all_sheets:
                all_sheets[new_name] = []  # initialize list
            all_sheets[new_name].append(df)

# Concatenate each list of DataFrames into one
for sheet_name in all_sheets:
    all_sheets[sheet_name] = pd.concat(all_sheets[sheet_name], ignore_index=True)



In [16]:
df_sgt = all_sheets['SGT Descriptors']
# df_sgt.columns = df_sgt.iloc[0]
# df_sgt = df_sgt[1:]
# df_sgt = df_sgt.dropna(axis=1)

param_rename_map = {
    "Number of nodes": "Nodes",
    "Number of edges": "Edges",
    "Network diameter": "Diameter",
    "Average edge angle (degrees)": "Avg. E. Angle",
    "Median edge angle (degrees)": "Med. E. Angle",
    "Graph density": "GD",
    "Average degree": "AD",
    "Global efficiency": "GE",
    "Wiener Index": "WI",
    "Assortativity coefficient": "ASC",
    "Average clustering coefficient": "ACC",
    "Average betweenness centrality": "BC",
    "Average eigenvector centrality": "EC",
    "Average closeness centrality": "CC",
}

# Rename Columns
# Apply replacements in the "Parameter" column
if "parameter" in df_sgt.columns:
    df_sgt["parameter"] = df_sgt["parameter"].replace(param_rename_map)

In [19]:
# Ensure the value columns exist
value_cols = ["value-1", "value-2", "value-3", "value-4"]

if all(col in df_sgt.columns for col in value_cols):
    df_sgt["Avg."] = df_sgt[value_cols].astype(float).mean(axis=1)
    df_sgt["Std. Dev."] = df_sgt[value_cols].astype(float).std(axis=1)

,parameter,value-1,value-2,value-3,value-4,Material
0,Nodes,7.460000e+03,7.438000e+03,7.447000e+03,7.426000e+03,C1_A
1,Edges,1.149100e+04,1.144200e+04,1.146100e+04,1.141400e+04,C1_A
2,Avg. E. Angle,1.515130e+02,1.515480e+02,1.515260e+02,1.516180e+02,C1_A
3,Med. E. Angle,9.000000e+01,9.000000e+01,9.000000e+01,9.000000e+01,C1_A
4,AD,3.080700e+00,3.076630e+00,3.078020e+00,3.074060e+00,C1_A
5,Diameter,1.660000e+02,1.670000e+02,1.700000e+02,1.660000e+02,C1_A
6,GD,4.100000e-04,4.100000e-04,4.100000e-04,4.100000e-04,C1_A
7,GE,2.463000e-02,2.454000e-02,2.463000e-02,2.455000e-02,C1_A
8,WI,1.712889e+09,1.711324e+09,1.706540e+09,1.703648e+09,C1_A
9,ASC,9.030000e-03,9.740000e-03,1.531000e-02,1.310000e-02,C1_A


In [18]:
# Filter the relevant materials and parameters
materials = ['C1_A', 'C3_C1', 'D2_B', 'D4-D']
material_names = ['Sample C1', 'Sample C3', 'Sample D2', 'Sample D4']
parameters = ['Nodes', 'Edges', 'AD', 'Diameter', 'GD', 'BC', 'CC', 'EC']

# Filter and pivot
# filtered_df = df_sgt[df_sgt['Material'].isin(materials) & df_sgt['Parameter'].isin(parameters)]
pivot_avg = df_sgt.pivot(index='Material', columns='parameter', values='Avg.')
pivot_std = df_sgt.pivot(index='Material', columns='parameter', values='Std. Dev.')

# Ensure consistent parameter order
pivot_avg = pivot_avg[parameters]
pivot_std = pivot_std[parameters]

#print(filtered_df)
print(f"Avg Table:\n{pivot_avg}")

KeyError: 'Avg.'